In [1]:
!pip install -r requirements.txt
from datetime import datetime as dt
from pathlib  import Path
import pandas as pd
import numpy  as np
import uuid
import glob
import os 
import sys
sys.path.insert(1, "\\".join(str(Path(os.path.dirname(os.getcwd()))).split("\\")[:-2]) + "\\Projects\\FIFADash\\trojan\\data engine")
import dataPrep as dp


[notice] A new release of pip available: 22.2.1 -> 23.1.2
[notice] To update, run: /workspace\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.8_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
INDEX_STOCKER_FILE_NAME_COL         = 0
INDEX_STOCKER_INDEX_COL             = 1
INDEX_STOCKER_ID_COL                = 2
INDEX_STOCKER_ORG_COL               = 3
INDEX_STOCKER_PATTERNIZED_COL       = 4
INDEX_STOCKER_LAST_UPDATED_COL      = 5
INDEX_STOCKER_MAIN_PATH_COL         = 6
INDEX_STOCKER_PATH_COL              = 7
INDEX_STOCKER_EXEC_DURATION_COL     = 8

MAINs_STOCKER_ID_COL                 = 0
MAINs_STOCKER_MAIN_INDEX_NAME_COL    = 1
MAINs_STOCKER_ORG_COL                = 2
MAINs_STOCKER_CUTTED_COL             = 3
MAINs_STOCKER_MAIN_PATH_COL          = 4
MAINs_STOCKER_SINGLE_SUBS_PATH_COL   = 5
MAINs_STOCKER_SUBS_COUNT_COL         = 6
MAINs_STOCKER_STOCKED_DATE_COL       = 7

IDs_GROUP_ID                         = 0
IDs_GROUP_INDEX                      = 1
IDs_GROUP_ORG                        = 2
IDs_GROUP_MAIN_NAME                  = 3
IDs_GROUP_MAIN_PATH                  = 4
IDs_GROUP_SUB_PATH                   = 5

# MAINs Cutter

### All MAINs

In [13]:
dp.get_MAINs_to_cut()

['/workspace/data\\trojan\\data bank\\Human and Economic Freedom\\CATO Institute\\MAIN - Human and Economic Freedom.csv']

### Reading `MAINs_stocker`

In [14]:
dp.update_MAINs_stocker_SUBs_count()
MAINs_stocker = dp.get_MAINs_stocker()
MAINs_stocker

,ID,MAIN Index Name,Org Name,Cutted,MAIN Path,SINGLE SUBs Path,SUBs Count,Stocked Date


### Adding new MAINs, Updating MAINs Stocker and configuring `Cutted`

In [15]:
## First we check if any of our MAINs were never registered in the MAINs stocker. 
## If so, these fresh new MAINs are added to the stocker with their 'Cutted' status set to 'False'
## The code then checks if the MAIN is in the stocker then does two checks on them: 
## Is the last modification date greater (newer) then the stocking date, and also checks if
## If the MAIN has 0 SUBs. If any of these checks is 'True', the MAIN's 'Cutted' status is set 
## to 'False', and their stocking date is updated. 
for MAIN in dp.get_MAINs_to_cut():
    ## MAIN is not in the stocker, add it and by default the 'Cutted' status is set to 'False'
    if MAIN not in list(MAINs_stocker["MAIN Path"]):
        dp.add_MAIN_to_stocker(MAIN)
    
    ## It IS in the stocker, but is it outdated? 
    else:
        # Get its index number 
        index_number                   = list(MAINs_stocker["MAIN Path"]).index(MAIN)
        # Get last modification date of this MAIN
        last_modification_date_of_MAIN = dp.get_modification_date(MAIN)
        # Get date when this MAIN has been last stocked
        stocked_date_of_MAIN           = dt.strptime(MAINs_stocker["Stocked Date"][index_number], "%Y-%m-%d %H:%M:%S")
        
        ## Check if the last modification of this MAIN is beyond its stocked date, then this MAIN is outdated
        if last_modification_date_of_MAIN > stocked_date_of_MAIN:
            # so we first set its 'cutted' status to False
            dp.set_cutted_status_of_this_MAIN(MAIN, False)
            # Then we update its 'Stocked Date'
            dp.update_MAIN_stocking_date(MAIN)
        
        # Let's also check if these MAINs have some SUBs
        elif dp.get_SUBs_Count_of_this_MAIN(MAIN) == 0:
            # If not, then we first set its 'cutted' status to False
            dp.set_cutted_status_of_this_MAIN(MAIN, False)
            # Then we update its 'Stocked Date'
            dp.update_MAIN_stocking_date(MAIN)

dp.get_MAINs_stocker()

,ID,MAIN Index Name,Org Name,Cutted,MAIN Path,SINGLE SUBs Path,SUBs Count,Stocked Date
0,240cf560,Human and Economic Freedom,CATO Institute,False,/workspace/data\trojan\...,NaN,0,2023-06-08 15:51:31


In [17]:
dp.get_index_stocker()

,File Name,Index,ID,Org Name,Patternized,Last Updated,MAIN Path,Path,Execution Duration


### Selecting need-to-be `Cutted` MAINs

In [18]:
MAINs_stocker          = dp.get_MAINs_stocker()
MAINs_to_cut_No_SUBs   = [] # list will be filled with MAINs that have no recorded SUBs
MAINs_to_cut_with_SUBs = [] # list will be filled with MAINs that HAVE SUBs

## This loop will browse all registered MAINs and picks out only those with 'Cutted' status set to 'False',
## and stage them for cutting
for i in range(len(MAINs_stocker["MAIN Path"])):
    if MAINs_stocker.iloc[i, MAINs_STOCKER_SUBS_COUNT_COL] == 0:
        MAINs_to_cut_No_SUBs.append(i)
    elif MAINs_stocker.iloc[i, MAINs_STOCKER_SUBS_COUNT_COL] > 0:
        if not MAINs_stocker.iloc[i, MAINs_STOCKER_CUTTED_COL]:
            MAINs_to_cut_with_SUBs.append(i)

## Now we have a list of MAINs to cut
MAINs_to_cut_with_SUBs

[]

In [19]:
MAINs_to_cut_No_SUBs

[0]

### Dealing with IDs

In [20]:
MAINs_IDs = []
for MAIN_index_number in MAINs_to_cut_with_SUBs:
    #IDs1 = []
    IDs2 = []
    
    ### Method 1
    ## That means we need to collect these SUBs' IDs
    # So we take a slice from the `index_stocker` such that the "MAIN Path" in `index_stocker` is equal to the MAIN path of the MAIN_index_number
    # that way we can isolate the SUBs and thus extract their IDs
    #index_stocker = dp.slice_df(dp.get_index_stocker(), MAINs_stocker.iloc[MAIN_index_number, MAINs_STOCKER_MAIN_PATH_COL], "MAIN Path")
    #
    # We then go over the column, extracting every ID substring from each value
    #for i in index_stocker["Index"]:
    #    IDs1.append(i.split("----")[INDEX_STOCKER_FILE_NAME_COL][:8])

    ### Method 2
    MAIN_Path = MAINs_stocker.iloc[MAIN_index_number, MAINs_STOCKER_MAIN_PATH_COL]
    MAIN_SINGLE_SUBs_Path = "\\".join(MAIN_Path.split("\\")[:-1]) + "\\SINGLE_SUBs\\"
    if os.path.exists(MAIN_SINGLE_SUBs_Path):
        Index_SINGLE_SUB_files = glob.glob(MAIN_SINGLE_SUBs_Path + "/*Index_SINGLE_SUB.csv")
        if len(Index_SINGLE_SUB_files) > 0:
            for SINGLE_SUB in Index_SINGLE_SUB_files:
                SINGLE_SUB = SINGLE_SUB.split("\\")[-1]
                IDs2.append(SINGLE_SUB.split("----")[1][:8])
    
    MAINs_IDs.append(IDs2)
MAINs_IDs

[]

### Cutting MAINs

In [21]:
IDs_index_names_stocker_from_MAINs = []

## Looping over MAINs (index numbers) that have SUBs, not creating new IDs
if len(MAINs_IDs) == len(MAINs_to_cut_with_SUBs):
    for i in range(len(MAINs_to_cut_with_SUBs)):
        ## Getting the MAIN's path
        MAIN_in_the_loop = MAINs_stocker.iloc[MAINs_to_cut_with_SUBs[i], 4]
        ## Cutting the MAIN while providing the IDs list, saving results in list
        IDs_index_names_stocker_from_MAINs.append(dp.MAINs_cutter(MAIN_in_the_loop, passed_IDs = MAINs_IDs[i]))
        ## Setting `Cutted` of this MAIN to True
        dp.set_cutted_status_of_this_MAIN(MAIN_in_the_loop, True)
else:
    print("Impossible Exception Reached")

## Looping over MAINs (index numbers) that have no SUBs (new MAINs), creating new IDs
for i in range(len(MAINs_to_cut_No_SUBs)):
    ## Getting the MAIN's path
    MAIN_in_the_loop = MAINs_stocker.iloc[MAINs_to_cut_No_SUBs[i], 4]
    ## Cutting the MAIN, saving results in list
    IDs_index_names_stocker_from_MAINs.append(dp.MAINs_cutter(MAIN_in_the_loop))
    ## Setting `Cutted` of this MAIN to True
    dp.set_cutted_status_of_this_MAIN(MAIN_in_the_loop, True)

### Updating Index Stocker

In [22]:
for IDs_group in IDs_index_names_stocker_from_MAINs:
    MAIN_in_the_loop = list(set(IDs_group["MAIN Path"]))[0]
    
    ## Checking if MAIN is in to index_stocker
    # if so, that means it has SUBs in the index_stocker
    if MAIN_in_the_loop in list(dp.get_index_stocker().iloc[:, INDEX_STOCKER_MAIN_PATH_COL]):
        ## filtering index_stocker to get the slice about the MAIN we are dealing with
        MAIN_sliced_index_stocker = dp.slice_df(dp.get_index_stocker(), MAIN_in_the_loop, "MAIN Path")
        
        ## We only keep the newly cut indexes and edit their info (Index, ID, Org, MAIN Path), and discard the rest
        discard_these_file_names = []
        IDs                      = []
        file_names               = []

        for sub in IDs_group["ID"]:
            i = 0
            for full in MAIN_sliced_index_stocker.iloc[:, INDEX_STOCKER_FILE_NAME_COL]:
                
                if sub in full:
                    IDs.append(sub)
                    file_names.append(full)
                    break
                i += 1

            if i == len(MAIN_sliced_index_stocker.iloc[:, INDEX_STOCKER_FILE_NAME_COL]):
                discard_these_file_names.append(full)

        ## edit info for this row
        for i in range(len(IDs)):
            index_index_number = list(dp.get_index_stocker().iloc[:, INDEX_STOCKER_FILE_NAME_COL]).index(file_names[i])
            index_path         = dp.get_index_stocker().iloc[index_index_number, INDEX_STOCKER_PATH_COL]
            org_name           = IDs_group.iloc[list(IDs_group["ID"]).index(IDs[i]), IDs_GROUP_ORG]
            index_name         = IDs_group.iloc[list(IDs_group["ID"]).index(IDs[i]), IDs_GROUP_INDEX]
            MAIN_Path          = list(set(IDs_group["MAIN Path"]))[0]

            dp.set_ID_of_this_index(index_path, IDs[i])
            dp.set_org_name_of_this_index(index_path, org_name)
            dp.set_Index_name_of_this_index(index_path, index_name)
            dp.set_MAIN_Path_of_this_index(index_path, MAIN_Path)
        
        discard_these_indexes = []
        ## discard these duplicated indexes
        for file_name in discard_these_file_names:
            print("----")
            print("REMOVING:")
            print(file_name)
            print(dp.get_index_stocker().iloc[list(dp.get_index_stocker().iloc[:, INDEX_STOCKER_FILE_NAME_COL]).index(file_name), INDEX_STOCKER_INDEX_COL])
            discard_these_indexes.append(list(dp.get_index_stocker().iloc[:, INDEX_STOCKER_FILE_NAME_COL]).index(file_name))
        dp.delete_list_indexes_from_index_stocker(list(set(discard_these_indexes)))
    
    # Otherwise, we are dealing with MAINs that are not in the index stocker, so we need to add the MAINs' SUBs into it
    else:
        for i in range(len(IDs_group)):
            SUB_index_path  = IDs_group.iloc[i, IDs_GROUP_SUB_PATH]
            index_ID        = IDs_group.iloc[i, IDs_GROUP_ID]
            index_Org       = IDs_group.iloc[i, IDs_GROUP_ORG]
            index_MAIN_path = IDs_group.iloc[i, IDs_GROUP_MAIN_PATH]
            index_name      = IDs_group.iloc[i, IDs_GROUP_INDEX]
            index_MAIN_name = IDs_group.iloc[i, IDs_GROUP_MAIN_NAME]

            dp.add_index_to_stocker(SUB_index_path)
            dp.set_ID_of_this_index(SUB_index_path, index_ID)
            dp.set_org_name_of_this_index(SUB_index_path, index_Org)
            dp.set_Index_name_of_this_index(SUB_index_path, index_name)
            dp.set_MAIN_Path_of_this_index(SUB_index_path, index_MAIN_path)

dp.get_index_stocker()

/workspace/data\trojan\data engine\dataPrep.py:410: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  new_index_stocker["Last Updated"]       = last_update
/workspace/data\trojan\data engine\dataPrep.py:410: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  new_index_stocker["Last Updated"]       = last_update


,File Name,Index,ID,Org Name,Patternized,Last Updated,MAIN Path,Path,Execution Duration
0,CATO Institute----25b071d2 Index_SINGLE_SUB.csv,HUMAN FREEDOM,25b071d2,CATO Institute,False,2023-06-08 15:52:10,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
1,CATO Institute----489b81f2 Index_SINGLE_SUB.csv,HUMAN FREEDOM RANK,489b81f2,CATO Institute,False,2023-06-08 15:52:10,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
2,CATO Institute----e19a4a4a Index_SINGLE_SUB.csv,HUMAN FREEDOM QUARTILE,e19a4a4a,CATO Institute,False,2023-06-08 15:52:10,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
3,CATO Institute----bbd6fc00 Index_SINGLE_SUB.csv,Ai Procedural Justice,bbd6fc00,CATO Institute,False,2023-06-08 15:52:10,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
4,CATO Institute----117cc24f Index_SINGLE_SUB.csv,Aii Civil Justice,117cc24f,CATO Institute,False,2023-06-08 15:52:10,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
...,...,...,...,...,...,...,...,...,...
133,CATO Institute----43a150e9 Index_SINGLE_SUB.csv,Eiiif Tax compliance,43a150e9,CATO Institute,False,2023-06-08 15:52:11,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
134,CATO Institute----7b63c2f9 Index_SINGLE_SUB.csv,Eiii Business regulations,7b63c2f9,CATO Institute,False,2023-06-08 15:52:11,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
135,CATO Institute----2ad870f4 Index_SINGLE_SUB.csv,E Regulation,2ad870f4,CATO Institute,False,2023-06-08 15:52:11,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN
136,CATO Institute----fe88e2fe Index_SINGLE_SUB.csv,ECONOMIC FREEDOM,fe88e2fe,CATO Institute,False,2023-06-08 15:52:11,/workspace/data\trojan\...,/workspace/data\trojan\...,NaN


### Validating Index Stocker

In [37]:
#print("Cost of Living:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[0])))  - 2))
#print("Health Care:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[1])))     - 2))
#print("Pollution:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[2])))       - 2))
#print("Property Price:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[3])))  - 2))
#print("Quality of Life:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[4]))) - 2))
#print("Traffic:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[5])))         - 2))
#print("Peace:", str(len(list(pd.read_csv(dp.get_MAINs_to_cut()[6])))           - 3))

Cost of Living: 7
Health Care: 3
Pollution: 139
Property Price: 3
Quality of Life: 8
Traffic: 10
Peace: 5


In [38]:
for MAIN in dp.get_MAINs_to_cut():
    SUBs_count = len(dp.slice_df(dp.get_index_stocker(), MAIN, "MAIN Path"))
    print(MAIN.split("\\")[-1][7:], "\t", str(SUBs_count))

Cost of Living.csv 	 7
Health Care.csv 	 3
Human and Economic Freedom.csv 	 138
Pollution.csv 	 3
Property Prices.csv 	 8
Quality of Life.csv 	 10
Traffic.csv 	 6
Global Peace + Positive Peace Indexes.csv 	 38


# Reset `stocker_index.csv`

In [5]:
dp.reset_index_stocker()

,File Name,Index,ID,Org Name,Patternized,Last Updated,MAIN Path,Path,Execution Duration


In [22]:
dp.get_index_stocker().head()

,File Name,Index,ID,Org Name,Patternized,Last Updated,MAIN Path,Path,Execution Duration
0,Global Peace + Positive Peace Index----080e7aa...,Global Terrorism Index,080e7aa0,Global Peace + Positive Peace Index,True,2023-06-08 22:29:41.817459,/workspace/data\trojan\...,/workspace/data\trojan\...,3:19:22.237634
1,Global Peace + Positive Peace Index----083a0ad...,Peacefulness Index,083a0ad6,Global Peace + Positive Peace Index,True,2023-06-09 01:59:00.018572,/workspace/data\trojan\...,/workspace/data\trojan\...,3:29:08.313329
2,Global Peace + Positive Peace Index----0bea2a3...,Safety & Security,0bea2a31,Global Peace + Positive Peace Index,True,2023-06-09 05:32:20.539037,/workspace/data\trojan\...,/workspace/data\trojan\...,3:32:48.613284
3,Global Peace + Positive Peace Index----1addb5b...,Militarisation,1addb5b0,Global Peace + Positive Peace Index,True,2023-06-09 08:59:41.853009,/workspace/data\trojan\...,/workspace/data\trojan\...,3:27:11.582599
4,Global Peace + Positive Peace Index----2f99cb2...,Ongoing Conflict,2f99cb2a,Global Peace + Positive Peace Index,True,2023-06-09 12:27:24.209186,/workspace/data\trojan\...,/workspace/data\trojan\...,3:27:10.796232


In [6]:
dp.reset_MAINs_stocker()

,ID,MAIN Index Name,Org Name,Cutted,MAIN Path,SINGLE SUBs Path,SUBs Count,Stocked Date


In [9]:
dp.get_MAINs_stocker()

,ID,MAIN Index Name,Org Name,Cutted,MAIN Path,SINGLE SUBs Path,SUBs Count,Stocked Date
0,0c3328de,Global Peace + Positive Peace Indexes,Global Peace + Positive Peace Index,True,/workspace/data\trojan\...,/workspace/data\trojan\...,38,2023-06-08 18:46:25
1,e750725c,Traffic,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,6,2023-06-08 18:46:25
2,a7e2043c,Quality of Life,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,10,2023-06-08 18:46:25
3,03168c6c,Property Prices,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,8,2023-06-08 18:46:25
4,006f9c4c,Pollution,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,3,2023-06-08 18:46:25
5,255d00e7,Human and Economic Freedom,CATO Institute,True,/workspace/data\trojan\...,NaN,0,2023-06-08 18:46:25
6,f4cea7ba,Health Care,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,3,2023-06-08 18:46:25
7,4b440c77,Cost of Living,Numbeo,True,/workspace/data\trojan\...,/workspace/data\trojan\...,7,2023-06-08 18:46:25


In [4]:
dp.get_candidate_indexes()

/workspace/data\trojan\data engine\dataPrep.py:410: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  new_index_stocker["Last Updated"]       = last_update
/workspace/data\trojan\data engine\dataPrep.py:410: FutureWarning: Inferring datetime64[ns] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype=datetime64[ns])
  new_index_stocker["Last Updated"]       = last_update


New index candidate found: ' Numbeo----03e3f9d0 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----51a03fbb Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----9f1e5492 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----aaa2300e Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----b23411b8 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----bd864cb0 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----cfdab287 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----49339c34 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----7931d502 Index_SINGLE_SUB.csv '
New index candidate found: ' Numbeo----80d25b31 Index_SINGLE_SUB.csv '
New index candidate found: ' CATO Institute----00765127 Index_SINGLE_SUB.csv '
New index candidate found: ' CATO Institute----0357fa4b Index_SINGLE_SUB.csv '
New index candidate found: ' CATO Institute----09de19c9 Index_SINGLE_SUB.csv '
New index candidate found: ' CATO Institute----0dfaf8

['/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----03e3f9d0 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----51a03fbb Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----9f1e5492 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----aaa2300e Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----b23411b8 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----bd864cb0 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Cost of Living\\Numbeo\\SINGLE_SUBs\\Numbeo----cfdab287 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Health Care\\Numbeo\\SINGLE_SUBs\\Numbeo----49339c34 Index_SINGLE_SUB.csv',
 '/workspace/data\\trojan\\data bank\\Health Care\\Numbeo\\SINGLE_S